# 📊 Phase 2: NSL-KDD Data Exploration
**Dataset**: NSL-KDD — Network Intrusion Detection

Attack categories:
| Category | Description |
|----------|-------------|
| **Normal** | Benign traffic |
| **DoS** | Denial of Service (neptune, smurf, teardrop…) |
| **PortScan** | Reconnaissance (nmap, ipsweep, portsweep…) |
| **BruteForce** | R2L attacks (guess_passwd, ftp_write…) |
| **U2R** | Privilege escalation (rootkit, buffer_overflow…) |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Style
plt.rcParams['figure.facecolor'] = '#0d1117'
plt.rcParams['axes.facecolor']   = '#161b22'
plt.rcParams['axes.labelcolor']  = '#c9d1d9'
plt.rcParams['xtick.color']      = '#c9d1d9'
plt.rcParams['ytick.color']      = '#c9d1d9'
plt.rcParams['text.color']       = '#c9d1d9'
plt.rcParams['grid.color']       = '#21262d'

PALETTE = {'Normal':'#2ecc71','DoS':'#e74c3c','PortScan':'#f39c12',
           'BruteForce':'#9b59b6','U2R':'#e91e63','Other':'#95a5a6'}

# Load datasets
train = pd.read_csv('../data/processed/nslkdd_train.csv')
test  = pd.read_csv('../data/processed/nslkdd_test.csv')

print(f'Train shape: {train.shape}')
print(f'Test  shape: {test.shape}')
train.head()

## 1️⃣ Attack Category Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.patch.set_facecolor('#0d1117')

for ax, (df, title) in zip(axes, [(train,'Train Set'),(test,'Test Set')]):
    counts = df['attack_category'].value_counts()
    colors = [PALETTE.get(c,'#95a5a6') for c in counts.index]
    bars = ax.bar(counts.index, counts.values, color=colors, edgecolor='#30363d', linewidth=0.8)
    ax.set_title(f'🔍 {title} — Attack Distribution', color='white', fontsize=13, pad=10)
    ax.set_xlabel('Attack Category')
    ax.set_ylabel('Count')
    ax.spines['bottom'].set_color('#30363d')
    ax.spines['left'].set_color('#30363d')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='y', alpha=0.3)
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
                f'{val:,}', ha='center', va='bottom', fontsize=9, color='#c9d1d9')

plt.tight_layout()
plt.savefig('../logs/attack_distribution.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()

## 2️⃣ Feature Types Overview

In [ ]:
print('Feature types:')
print(train.dtypes.value_counts())
print('\nCategorical features:')
cat_cols = train.select_dtypes(include='object').columns.tolist()
print(cat_cols)
print('\nNumerical features:')
num_cols = train.select_dtypes(include=np.number).columns.tolist()
print(num_cols)

## 3️⃣ Protocol × Attack Heatmap

In [ ]:
ct = pd.crosstab(train['protocol_type'], train['attack_category'])

fig, ax = plt.subplots(figsize=(10, 4))
fig.patch.set_facecolor('#0d1117')
sns.heatmap(ct, annot=True, fmt='d', cmap='YlOrRd', ax=ax,
            linewidths=0.5, linecolor='#30363d',
            cbar_kws={'label': 'Count'})
ax.set_title('Protocol × Attack Category', color='white', fontsize=13)
ax.set_xlabel('Attack Category')
ax.set_ylabel('Protocol')
plt.tight_layout()
plt.savefig('../logs/protocol_attack_heatmap.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()

## 4️⃣ Top Numeric Features — Box Plot by Category

In [ ]:
top_features = ['src_bytes','dst_bytes','duration','count','srv_count','same_srv_rate']

fig, axes = plt.subplots(2, 3, figsize=(18, 8))
fig.patch.set_facecolor('#0d1117')
axes = axes.flatten()

for i, feat in enumerate(top_features):
    ax = axes[i]
    for cat, color in PALETTE.items():
        subset = train[train['attack_category'] == cat][feat].clip(upper=train[feat].quantile(0.99))
        if len(subset) > 0:
            ax.hist(subset, bins=40, alpha=0.6, color=color, label=cat, density=True)
    ax.set_title(feat, color='white', fontsize=11)
    ax.set_xlabel('Value')
    ax.set_ylabel('Density')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_color('#30363d')
    ax.spines['left'].set_color('#30363d')
    ax.grid(alpha=0.2)

patches = [mpatches.Patch(color=c, label=l) for l, c in PALETTE.items()]
fig.legend(handles=patches, loc='lower center', ncol=5, frameon=False,
           bbox_to_anchor=(0.5, -0.02))
plt.suptitle('Feature Distributions by Attack Category', color='white', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../logs/feature_distributions.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()

## 5️⃣ Correlation Matrix (Numeric Features)

In [ ]:
corr = train[num_cols[:20]].corr()

fig, ax = plt.subplots(figsize=(14, 10))
fig.patch.set_facecolor('#0d1117')
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='coolwarm', center=0, vmin=-1, vmax=1,
            annot=False, ax=ax, linewidths=0.3, linecolor='#21262d')
ax.set_title('Feature Correlation Matrix', color='white', fontsize=13)
plt.tight_layout()
plt.savefig('../logs/correlation_matrix.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()

print('\n✅ Data Exploration Complete!')
print('   → Next: notebooks/03_preprocessing.ipynb')